# Lab06: Writing Data in Neo4j

Santiago Elí Jiménez Aguilar  
Luis Eduardo Gonzalez Gloria

## Goal:
Create a data pipeline to analyze the Recommendation Videogames dataset <br>
(https://networkrepository.com/rec-amz-Video-Games.php).

## Instructions
- **Dataset**. Download the Recommendation Videogames dataset from the Network Repository.
- **Data Ingestion**. This section should contain a code cell using PySpark to read the DataFrame.
- **Graph Analysis**. This section should contain the code to generate:
  - PageRank
  - Label Propagation
  - Triangle Counting
  - Degree Distribution
- **Writing Data in Neo4j**. This section should contain the code to persist the nodes and edges DataFrames in Neo4j.
- **Querying the Graph**. This section should contain a screenshot of the graph written to Neo4j.

In [1]:
from spark_utils import SparkUtils
neo4j_connector = "org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3,io.graphframes:graphframes-spark3_2.13:0.9.0-spark3.5"
su = SparkUtils("Lab06: Writing Data in Neo4j", "spark://spark-master:7077", spark_packages=neo4j_connector)
su.spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.neo4j#neo4j-connector-apache-spark_2.13 added as a dependency
io.graphframes#graphframes-spark3_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-99119602-5644-4c44-82c6-27608876d78f;1.0
	confs: [default]
	found org.neo4j#neo4j-connector-apache-spark_2.13;5.3.10_for_spark_3 in central
	found org.neo4j#neo4j-connector-apache-spark_2.13_common;5.3.10_for_spark_3 in central
	found org.neo4j#caniuse-core;1.3.0 in central
	found org.neo4j#caniuse-api;1.3.0 in central
	found org.jetbrains.kotlin#kotlin-stdlib;2.1.20 in central
	found org.jetbrains#annotations;13.0 in central
	found org.neo4j#caniuse-neo4j-detection;1.3.0 in central
	found org.neo4j.driver#neo4j-java-driver-slim;4.4.21 in central
	found org.reactivestreams#reactiv

## 1. Data Ingestion

In [2]:
from graphframes import GraphFrame
from pyspark.sql import functions as F

mvideo_games_schema = SparkUtils.generate_schema([
    ("userId",      "string"),
    ("videoGameId", "string"),
    ("rating",      "float"),
    ("timestamp",   "long")
])

video_games_df = (su.spark.read
                .option("header", "false")
                .schema(mvideo_games_schema)
                .csv("/opt/spark/work-dir/data/rec-amz-Video-Games"))

video_games_df = video_games_df.filter(
    ~F.col("userId").startswith("<") &
    ~F.col("videoGameId").startswith("<")
)

# 1. User vertices
user_vertices = video_games_df.select(
    F.col("userId").alias("id"),
    F.lit("user").alias("type")
).distinct()

# 2. Game vertices
game_vertices = video_games_df.select(
    F.col("videoGameId").alias("id"),
    F.lit("game").alias("type")
).distinct()

# 3. Union both vertex types
vertices = user_vertices.union(game_vertices)

# 4. Edges
edges = video_games_df.select(
    F.col("userId").alias("src"),
    F.col("videoGameId").alias("dst"),
    F.col("rating").alias("rating"),
    F.col("timestamp").alias("timestamp")
)

# 5. Create the GraphFrame
g = GraphFrame(vertices, edges)
g.vertices.show()
g.edges.show()

/usr/local/lib/python3.10/dist-packages/pyspark/sql/classic/dataframe.py:146: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
                                                                                

+--------------+----+
|            id|type|
+--------------+----+
| AE7GUHCDQQ4UI|user|
|A26B0P6K95SIKW|user|
|A182S3ANC0W7DL|user|
|A1T98OCCYW6OBI|user|
|A1TBUSGCBTXWFC|user|
|A366EUKI8WMGYB|user|
|A129SW886TYQ6H|user|
|A2M64UKVOU9CWU|user|
|A10N7L0GMRODUO|user|
| AFWPLXT2OD6H1|user|
|A2890J1FHE76RY|user|
| AJRHPTQ7TXPD6|user|
|A2NQXA21O64HDZ|user|
|A261CI99KET69W|user|
|A214Z566V6QEDF|user|
|A1NC9PQCE3NOGA|user|
|A27QRTHZLBA61M|user|
| ADOCLYEFV2PKH|user|
| A8SSL0QMV2VY1|user|
| AA9LU15A9PX9E|user|
+--------------+----+
only showing top 20 rows
+--------------+----------+------+----------+
|           src|       dst|rating| timestamp|
+--------------+----------+------+----------+
| AB9S9279OZ3QO|0078764343|   5.0|1373155200|
|A24SSUT5CSW8BH|0078764343|   5.0|1377302400|
| AK3V0HEBJMQ7J|0078764343|   4.0|1372896000|
|A10BECPH7W8HM7|043933702X|   5.0|1404950400|
|A2PRV9OULX1TWP|043933702X|   5.0|1386115200|
| AE7GUHCDQQ4UI|043933702X|   1.0|1366156800|
| A48ABFDDRMKI8|043933702X|   5.0

## 2. Graph Analysis

### PageRank

In [3]:
# maxIter=5 keeps runtime manageable on large datasets
results = g.pageRank(resetProbability=0.15, maxIter=5)

print("=== TOP USUARIOS POR PAGERANK ===")
results.vertices \
    .filter(F.col("type") == "user") \
    .select("id", "type", "pagerank") \
    .orderBy("pagerank", ascending=False) \
    .show(10)

print("=== TOP VIDEOJUEGOS POR PAGERANK ===")
results.vertices \
    .filter(F.col("type") == "game") \
    .select("id", "type", "pagerank") \
    .orderBy("pagerank", ascending=False) \
    .show(10)

print("=== ARISTAS CON PESO ===")
results.edges \
    .select("src", "dst", "rating", "weight") \
    .orderBy("weight", ascending=False) \
    .show(10)

/usr/local/lib/python3.10/dist-packages/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


=== TOP USUARIOS POR PAGERANK ===


+--------------------+----+------------------+
|                  id|type|          pagerank|
+--------------------+----+------------------+
|A0009878M2RGMMHGJH39|user|0.5551439694748398|
|A0002090WKEMAO8KOWKM|user|0.5551439694748398|
|A00063061AK7XBIZL...|user|0.5551439694748398|
|A00101847G3FJTWYGNQA|user|0.5551439694748398|
|A00101961G0VS92WD...|user|0.5551439694748398|
|A00278362652PLQL0...|user|0.5551439694748398|
|A00096001PYDTZQQ4...|user|0.5551439694748398|
|A00593941PIA0QVHH...|user|0.5551439694748398|
|A001147626R4BL248...|user|0.5551439694748398|
|A00933513755ZZ7RE...|user|0.5551439694748398|
+--------------------+----+------------------+
only showing top 10 rows
=== TOP VIDEOJUEGOS POR PAGERANK ===


+----------+----+------------------+
|        id|type|          pagerank|
+----------+----+------------------+
|B00DJFIMW6|game| 7285.526431395484|
|B00BGA9WK2|game| 2645.507195083908|
|B00FAX6XQC|game|2547.3341328377564|
|B009KS4XRO|game|2501.0385889444806|
|B0055SWM08|game|1989.4563258041183|
|B00CSR2J9I|game| 1964.373283273377|
|B002VBWIP6|game|1857.1934441848766|
|B0015AARJI|game|1408.3694893308366|
|B000FKBCX4|game|1243.5554664378442|
|B00178630A|game| 1230.274656615952|
+----------+----+------------------+
only showing top 10 rows
=== ARISTAS CON PESO ===


[Stage 419:======================================>                  (2 + 1) / 3]

+--------------------+----------+------+------+
|                 src|       dst|rating|weight|
+--------------------+----------+------+------+
|A008045111DKTFZLQ...|B002BRZ79E|   5.0|   1.0|
|A017699216H6YAFBG...|B000T5MS9C|   5.0|   1.0|
|A01206022Z1GUQ3KQ...|B003KT9ECI|   5.0|   1.0|
|A039992814X2DPIB1...|B005UI9BT8|   2.0|   1.0|
|A01416443H8V31K9R...|B0050SVGXM|   5.0|   1.0|
|A01882789KR2FROBMN8K|B008L3UUPS|   1.0|   1.0|
|A0233749EVODU302LHWD|B00CSR2J9I|   5.0|   1.0|
|A0009878M2RGMMHGJH39|B0015AARJI|   5.0|   1.0|
|A0099919A30Q9445KCR1|B00191GSYG|   5.0|   1.0|
|A001147626R4BL248...|B00BXONG7G|   2.0|   1.0|
+--------------------+----------+------+------+
only showing top 10 rows


### Label Propagation

In [4]:
lpa = g.labelPropagation(maxIter=3)
lpa.show()

[Stage 540:=============================================>           (4 + 1) / 5]

+--------------------+----+-----------+
|                  id|type|      label|
+--------------------+----+-----------+
|          043933702X|game|25769804420|
|          0439671418|game| 8589972610|
|          0439900581|game|17179871748|
|          0545115507|game|      55201|
|          0700026657|game| 8590005570|
|          1886846758|game|17179911574|
|          7293000936|game| 8590098049|
|          7542614444|game|25769958148|
|          9078439122|game|34359872010|
|          9861064222|game|17179908478|
|          986118452X|game| 8590070871|
|          9861767304|game| 8589942402|
|          986325083X|game|25769883582|
|          9941113300|game|      94401|
|A0009878M2RGMMHGJH39|user|     169495|
|A00101961G0VS92WD...|user| 8590100259|
|A001147626R4BL248...|user|25769977765|
|A0011756FPL8K71Q5TAQ|user|34359907769|
|A00230923E4Y7VHWZ...|user| 8590107813|
|A00338543M2OZPUWO...|user|     172460|
+--------------------+----+-----------+
only showing top 20 rows


### Triangle Counting

In [5]:
triangle_count = g.triangleCount()
triangle_count.show()

[Stage 665:=============================================>           (4 + 1) / 5]

+-----+--------------+----+
|count|            id|type|
+-----+--------------+----+
|    0| AE7GUHCDQQ4UI|user|
|    0|A1TBUSGCBTXWFC|user|
|    0|A129SW886TYQ6H|user|
|    0| AFWPLXT2OD6H1|user|
|    0|A26B0P6K95SIKW|user|
|    0|A366EUKI8WMGYB|user|
|    0|A10N7L0GMRODUO|user|
|    0|A261CI99KET69W|user|
|    0|A214Z566V6QEDF|user|
|    0|A182S3ANC0W7DL|user|
|    0|A2M64UKVOU9CWU|user|
|    0|A2890J1FHE76RY|user|
|    0| AJRHPTQ7TXPD6|user|
|    0|A1NC9PQCE3NOGA|user|
|    0|A27QRTHZLBA61M|user|
|    0| AA9LU15A9PX9E|user|
|    0|A1T98OCCYW6OBI|user|
|    0|A2NQXA21O64HDZ|user|
|    0| ADOCLYEFV2PKH|user|
|    0| A8SSL0QMV2VY1|user|
+-----+--------------+----+
only showing top 20 rows


### Degree Distribution

#### InDegree

In [6]:
in_deg = g.inDegrees.join(vertices, "id")
in_deg.show()

+----------+--------+----+
|        id|inDegree|type|
+----------+--------+----+
|0439394422|       2|game|
|7118021156|       2|game|
|986325083X|       1|game|
|9882077153|       7|game|
|B000006OWT|      11|game|
|B000006RGR|      28|game|
|B00000DMAI|      19|game|
|B00000I1BK|      44|game|
|B00000IFKW|       3|game|
|B00000IGZM|      13|game|
|B00000K13F|       1|game|
|B00000K2XJ|      26|game|
|B00000K4E7|       4|game|
|B00000K4KF|      26|game|
|B00000K4YE|       4|game|
|B00000K516|       5|game|
|B00000K51C|      10|game|
|B00001L5TD|       3|game|
|B00001N2MM|       1|game|
|B00001OX3R|       2|game|
+----------+--------+----+
only showing top 20 rows


#### OutDegree

In [7]:
out_deg = g.outDegrees.join(vertices, "id")
out_deg.show()

[Stage 696:>                                                        (0 + 1) / 1]

+--------------------+---------+----+
|                  id|outDegree|type|
+--------------------+---------+----+
|A0002090WKEMAO8KOWKM|        1|user|
|A00063061AK7XBIZL...|        1|user|
|A00089163LKXK4V19...|        1|user|
|A00096001PYDTZQQ4...|        2|user|
|A0009878M2RGMMHGJH39|        1|user|
|A00101847G3FJTWYGNQA|        3|user|
|A00101961G0VS92WD...|        1|user|
|A001147626R4BL248...|        1|user|
|A0011756FPL8K71Q5TAQ|        2|user|
|A001412229RSF8NQJ...|        2|user|
|A00158021VAJ9275D...|        1|user|
|A00160082V0HQVAXB...|        1|user|
|A00166281YWM98A3S...|        3|user|
|A00190541RH03G6MZ...|        1|user|
|A00230923E4Y7VHWZ...|        1|user|
|A002439424KGHR3LZ...|        1|user|
|A00278362652PLQL0...|        1|user|
|A0033772XS33OKK28RIW|        1|user|
|A00338543M2OZPUWO...|        1|user|
|A00416781KPR6K34D...|        1|user|
+--------------------+---------+----+
only showing top 20 rows


## 3. Writing Data in Neo4j

In [ ]:
neo4j_url    = "bolt://neo4j-iteso:7687"
neo4j_user   = "neo4j"
neo4j_passwd = "neo4j@1234"

# ── Write User nodes (:User) ──────────────────────────────────────────────────
g.vertices.filter(F.col("type") == "user").write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("labels", ":User") \
  .option("node.keys", "id") \
  .save()

# ── Write Game nodes (:Game) ──────────────────────────────────────────────────
g.vertices.filter(F.col("type") == "game").write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("labels", ":Game") \
  .option("node.keys", "id") \
  .save()

# ── Write edges (User)-[:RATED]->(Game) ──────────────────────────────────────
g.edges.repartition(8).write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("relationship", "RATED") \
  .option("relationship.save.strategy", "keys") \
  .option("relationship.source.labels", ":User") \
  .option("relationship.source.save.mode", "match") \
  .option("relationship.source.node.keys", "src:id") \
  .option("relationship.target.labels", ":Game") \
  .option("relationship.target.save.mode", "match") \
  .option("relationship.target.node.keys", "dst:id") \
  .option("batch.size", "5000") \
  .save()

print("Vertices and edges wrote in Neo4j")

[Stage 699:>                                                        (0 + 1) / 3]

## 4. Querying the Graph

> **Screenshot of the Neo4j graph goes here.**

In [ ]:
su.spark.stop()